In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [4]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_fern = df[df["Network ID"] == 11]

df_fern = df_fern.loc[:, df_fern.notna().any()]
df_fern_vanessa = df_fern[df_fern['Phone number'].str.contains("Vanessa", case=False, na=False)]

df_fern_not_vanessa = df_fern[~df_fern['Phone number'].str.contains("Vanessa", case=False, na=False)]

In [5]:
df_left = df_fern_not_vanessa[df_fern_not_vanessa['ISSUE'].notna()]

df_loc_change = df_left[df_left['ISSUE'].str.contains("Attribution, Location", case=False, na=False)]

df_loc_change_info = df_loc_change[['Station ID','Station_name', 'lat', 'lon', 'Elevation', 'native_id']]

df_loc_change_info

,Station ID,Station_name,lat,lon,Elevation,native_id
1977,13043.0,WADF1 Seedtree,49.6233,-117.0436,815,WR02CC
1978,13044.0,WADF2 Burn,49.6445,-117.0626,1270,WR01CC
1979,13045.0,WADF4 Cabin,49.6693,-117.0643,1715,WR05CC
1980,13046.0,WADF5 Alpine,49.69,-117.0865,2085,WR04AL


In [5]:
import sqlalchemy as sa
from sqlalchemy.orm import Session


engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


In [6]:
update_meta_history = sa.text("""
    UPDATE meta_history
    SET station_name = :new_name,
        lat = :new_lat,
        lon = :new_lon,
        elev = :new_elev
    WHERE station_id = :station_id
""")

update_meta_station = sa.text("""
    UPDATE meta_station
    SET native_id = :new_native_id
    WHERE station_id = :station_id
""")

with engine.begin() as conn:
    with conn.begin_nested():  # nested transaction (SAVEPOINT)
        for idx, row in df_loc_change_info.iterrows():
            station_id = row["Station ID"]

            # Fetch current values
            current_history = conn.execute(
                sa.text("SELECT station_name, lat, lon, elev FROM meta_history WHERE station_id = :station_id"),
                {"station_id": station_id}
            ).fetchone()

            current_station = conn.execute(
                sa.text("SELECT native_id FROM meta_station WHERE station_id = :station_id"),
                {"station_id": station_id}
            ).fetchone()

            print(f"Station ID: {station_id}")
            print(f"  meta_history -> station_name: {current_history[0]} -> {row['Station_name']}")
            print(f"                 lat: {current_history[1]} -> {row['lat']}")
            print(f"                 lon: {current_history[2]} -> {row['lon']}")
            print(f"                 elev: {current_history[3]} -> {row['Elevation']}")
            print(f"  meta_station -> native_id: {current_station[0]} -> {row['native_id']}")
            print("-" * 60)

            # Execute updates
            conn.execute(update_meta_history, {
                "new_name": row["Station_name"],
                "new_lat": row["lat"],
                "new_lon": row["lon"],
                "new_elev": row["Elevation"],
                "station_id": station_id
            })

            conn.execute(update_meta_station, {
                "new_native_id": row["native_id"],
                "station_id": station_id
            })

        # Rollback to preview only
        conn.rollback()
        print("Rollback complete. No changes were committed.")

Station ID: 13043.0
  meta_history -> station_name: WADF1 Seedtree -> WADF1 Seedtree
                 lat: 49.6233 -> 49.6233
                 lon: -117.0436 -> -117.0436
                 elev: 815 -> 815
  meta_station -> native_id: WR02CC -> WR02CC
------------------------------------------------------------
Station ID: 13044.0
  meta_history -> station_name: WADF2 Burn -> WADF2 Burn
                 lat: 49.6445 -> 49.6445
                 lon: -117.0626 -> -117.0626
                 elev: 1270 -> 1270
  meta_station -> native_id: WR01CC -> WR01CC
------------------------------------------------------------
Station ID: 13045.0
  meta_history -> station_name: WADF4 Cabin -> WADF4 Cabin
                 lat: 49.6693 -> 49.6693
                 lon: -117.0643 -> -117.0643
                 elev: 1715 -> 1715
  meta_station -> native_id: WR05CC -> WR05CC
------------------------------------------------------------
Station ID: 13046.0
  meta_history -> station_name: WADF5 Alpine -> WADF5 

In [25]:
### Real update

with engine.begin() as conn:  # this begins a real transaction
    for idx, row in df_loc_change_info.iterrows():
        station_id = row["Station ID"]

        # Execute updates
        conn.execute(update_meta_history, {
            "new_name": row["Station_name"],
            "new_lat": row["lat"],
            "new_lon": row["lon"],
            "new_elev": row["Elevation"],
            "station_id": station_id
        })

        conn.execute(update_meta_station, {
            "new_native_id": row["native_id"],
            "station_id": station_id
        })

# No need to call conn.commit() explicitly — using `engine.begin()` auto-commits if no exceptions
print("Update completed successfully.")

Update completed successfully.


### For the satation 12063 Renamed to Sunbeam (13069) Check for duplicate data - delete this entry after identified gaps are filled. 



In [8]:
query = """
SELECT o.obs_time, o.vars_id, o.datum, m.station_id
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN obs_raw AS o ON m.history_id = o.history_id
WHERE s.network_id = 11
  AND m.station_id IN (12063, 13069)
ORDER BY obs_time, vars_id, m.station_id
"""
df_obs = pd.read_sql(query, engine)

df_pivot = df_obs.pivot_table(
    index=['obs_time', 'vars_id'],
    columns='station_id',
    values='datum'
).reset_index()

df_pivot.columns.name = None  # remove the pivot name
df_pivot

,obs_time,vars_id,12063,13069
0,2009-09-09 16:00:00,502,794.250,794.250
1,2009-09-09 16:00:00,503,3.617,3.617
2,2009-09-09 16:00:00,504,90.500,90.500
3,2009-09-09 16:00:00,506,6.310,6.310
4,2009-09-09 16:00:00,507,10.020,10.020
...,...,...,...,...
820686,2024-01-08 12:00:00,505,NaN,-15.331
820687,2024-01-08 12:00:00,506,NaN,4.290
820688,2024-01-08 12:00:00,507,NaN,6.630
820689,2024-01-08 12:00:00,508,NaN,115.100


In [15]:
df_pivot[13069].isna().sum()

np.int64(0)

## Fill the gap, need to use insert

In [10]:
# Select only rows where 13069 is NaN and 12063 is not
df_fill = df_pivot[df_pivot[13069].isna() & df_pivot[12063].notna()]
df_fill

,obs_time,vars_id,12063,13069,same


In [ ]:

# # Example: get history_id for station 13069
# from sqlalchemy import text
# session.rollback()  # just to preview changes, nothing committed yet

# history_id_13069 = session.execute(
#     text("SELECT history_id FROM meta_history WHERE station_id = 13069")
# ).scalar()

# update_sql = text("""
#     UPDATE obs_raw
#     SET datum = :datum
#     WHERE history_id = :history_id AND obs_time = :obs_time AND vars_id = :vars_id
# """)

# for _, row in df_fill.iterrows():
#     session.execute(update_sql, {
#         "datum": row[12063],
#         "history_id": history_id_13069,
#         "obs_time": row["obs_time"],
#         "vars_id": row["vars_id"]
#     })

# # Check first
# session.rollback()  # just to preview changes, nothing committed yet
# # Once confirmed:
# # session.commit()

In [54]:
from sqlalchemy import text
from datetime import datetime
session.rollback()  # nothing is committed yet

# history_id_13069 = session.execute(
#     text("SELECT history_id FROM meta_history WHERE station_id = 13069")
# ).scalar()

insert_sql = text("""
INSERT INTO obs_raw (history_id, obs_time, vars_id, datum)
VALUES (:history_id, :obs_time, :vars_id, :datum)
""")

for _, row in df_fill.iloc[30:31].iterrows():
    session.execute(insert_sql, {
        "history_id": history_id_13069,
        "obs_time": row["obs_time"],
        "vars_id": row["vars_id"],
        "datum": row[12063]
    })

# Preview changes without committing

# Once confirmed, commit
session.commit()

In [16]:
### The other difference should be caused by different round

# Keep only rows where both stations have non-NaN values
mask_not_nan = df_pivot[12063].notna() & df_pivot[13069].notna()

# Keep only rows where the values are different
mask_diff = df_pivot[12063] != df_pivot[13069]

# Combine masks
df_mismatch = df_pivot[mask_not_nan & mask_diff]

df_mismatch

,obs_time,vars_id,12063,13069,same
285751,2016-10-04 14:00:00,506,3.274,3.27,False
285752,2016-10-04 14:00:00,507,8.814,8.81,False
285753,2016-10-04 14:00:00,508,101.082,101.10,False
285754,2016-10-04 14:00:00,509,135.625,135.60,False
285760,2016-10-04 15:00:00,506,3.777,3.78,False
...,...,...,...,...,...
340834,2017-06-16 14:00:00,509,479.375,479.40,False
340841,2017-06-16 15:00:00,507,6.044,6.04,False
340843,2017-06-16 15:00:00,509,766.875,766.90,False
340850,2017-06-16 16:00:00,507,7.554,7.55,False


### Delete the station 12063, McBridePeak

In [ ]:
from sqlalchemy import text

# delete_obs = text("""
#     DELETE FROM obs_raw
#     WHERE history_id IN (
#         SELECT history_id FROM meta_history
#         WHERE station_id = :sid
#     );
# """)

# delete_history = text("""
#     DELETE FROM meta_history
#     WHERE station_id = :sid;
# """)

delete_station = text("""
    DELETE CASCADE FROM meta_station
    WHERE station_id = :sid;
""")

In [ ]:
## Preview first (no real deletion)
sid = 12063   # the station you want to delete

with session.begin_nested():   # create SAVEPOINT
    session.execute(delete_obs, {"sid": sid})
    session.execute(delete_history, {"sid": sid})
    session.execute(delete_station, {"sid": sid})

    print("Preview only — changes NOT saved.")
    # Query to show what would be deleted:
    remaining = session.execute(
        text("SELECT * FROM meta_station WHERE station_id = :sid"),
        {"sid": sid}
    ).fetchall()
    print("Remaining station rows (should be empty in preview):", remaining)

    # Rollback explicitly, but begin_nested() will auto-rollback anyway
    session.rollback()

In [6]:
sid = 12063

with session.begin():   # real transaction
    session.execute(delete_obs, {"sid": sid})
    session.execute(delete_history, {"sid": sid})
    session.execute(delete_station, {"sid": sid})

print("Station 12063 fully deleted.")

Station 12063 fully deleted.


In [15]:
2084-1967

117

In [7]:
### verify deletion
check = session.execute(
    text("SELECT * FROM meta_station WHERE station_id = 12063")
).fetchall()

print("Should be empty:", check)

Should be empty: []


In [12]:
session.rollback()   # very important — reset transaction

check = session.execute(
    text("SELECT * FROM meta_station WHERE native_id = 'SpectrumWx'")
).fetchall()

print(check)

[]
